Исходя из данных результатов, наилучшие значения ML-метрик показала модель Unet с efficientnet-b0. Это может объясняться небольшим количеством данных, т.к. для трансформерных моделей и моделей DeepLabV3 необходимо большее количество данных.
В качестве улучшенной модели был применен тот же трансформер, оптимизированный методом QLoRA с целью получения лучших метрик на данной архитектуре.

#### Улучшенная модель Unet с трансформерным энкодером mit-b1 с применением метода QLoRA

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch.nn.functional as F

In [ ]:
# 1. Настройка 4-битной квантизации для экономии памяти
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Загрузка предобученной модели. 'nvidia/mit-b1' — пример, уточните ID
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    quantization_config=bnb_config,
    device_map='auto',
    ignore_mismatched_sizes=True
)

# 3. Настройка классификатора под 1 класс
model.decode_head.classifier = torch.nn.Conv2d(256, 1, kernel_size=1)

# 4. Заморозка основной модели
for param in model.parameters():
    param.requires_grad = False

# 5. Настройка LoRA
lora_config = LoraConfig(
    r=8,                      # Ранг матриц адаптации
    lora_alpha=32,            # Коэффициент масштабирования
    target_modules=["q_proj", "v_proj"], # Целевые слои в трансформере
    lora_dropout=0.1,         # Dropout для регуляризации
    bias="none"
)

# 5. Применение LoRA к модели
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.to(device)

In [ ]:
class FUSegSegformerDataset(Dataset):
    def __init__(self, root, split='train', processor_model_id="nvidia/segformer-b0-finetuned-ade-512-512", 
                 geometric_augmentations=None):
        """
        Args:
            root: путь к корневой папке датасета (содержит train, validation, test)
            split: 'train', 'validation' или 'test'
            processor_model_id: ID модели SegFormer для загрузки процессора
            geometric_augmentations: композиция albumentations только с геометрическими преобразованиями 
                                     (без нормализации и изменения цвета). Если None, применяются стандартные.
        """
        self.root = Path(root)
        self.split = split
        self.img_dir = self.root / split / 'images'
        self.mask_dir = self.root / split / 'labels'
        
        self.imgs = sorted(list(self.img_dir.glob('*.png')))
        self.masks = [self.mask_dir / img.name for img in self.imgs]
        
        # Загружаем процессор SegFormer
        self.processor = SegformerImageProcessor.from_pretrained(processor_model_id)
        
        # Геометрические аугментации (по умолчанию: горизонтальное отражение и лёгкий поворот/масштаб)
        if geometric_augmentations is None and split == 'train':
            self.geometric_aug = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Affine(scale=(0.85, 1.15), rotate=20, translate_percent=0.05, p=0.5)
            ])
        elif geometric_augmentations is not None:
            self.geometric_aug = geometric_augmentations
        else:
            self.geometric_aug = None
    
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        # Загружаем изображение и маску
        img_path = self.imgs[idx]
        mask_path = self.masks[idx]
        
        img = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        mask = (np.array(mask) > 127).astype(np.uint8)
        
        # Применяем геометрические аугментации (если есть)
        if self.geometric_aug is not None:
            augmented = self.geometric_aug(image=np.array(img), mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        
        # Используем процессор для ресайза, нормализации и преобразования в тензоры
        encoding = self.processor(images=img, segmentation_maps=mask, return_tensors="pt")
        
        # Убираем лишнее измерение (batch)
        pixel_values = encoding['pixel_values'].squeeze()  # (3, H, W)
        labels = encoding['labels'].squeeze().long()       # (H, W)
        
        return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
train_ds = FUSegSegformerDataset(DATASET_ROOT, 'train', processor_model_id="nvidia/segformer-b0-finetuned-ade-512-512")
val_ds   = FUSegSegformerDataset(DATASET_ROOT, 'validation', processor_model_id="nvidia/segformer-b0-finetuned-ade-512-512")
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
criterion = DiceFocalLoss(dice_weight=1.0, focal_weight=0.5, focal_gamma=2.0, focal_alpha=0.25)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

In [ ]:
random.seed(42)


best_model_path = WEIGHTS_DIR / "baseline_vit_best.pth"

best_dice = 0.0
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, 16):
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values)
        logits = outputs.logits
        logits = F.interpolate(logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
        loss = criterion(logits, labels.float().unsqueeze(1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Валидация
    model.eval()
    val_loss = 0.0
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            logits = F.interpolate(logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(logits, labels.float().unsqueeze(1))
            val_loss += loss.item()
            
            pred = torch.sigmoid(logits) > 0.5
            pred = pred.float()
            tp += (pred * labels).sum().item()
            fp += (pred * (1 - labels)).sum().item()
            fn += ((1 - pred) * labels).sum().item()
            tn += ((1 - pred) * (1 - labels)).sum().item()
    
    val_loss /= len(val_loader)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou  = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    
    scheduler.step(dice)

    
    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")
    if dice > best_dice:
      best_dice = dice
      best_val_loss = val_loss
      torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': dice,
            'val_loss': val_loss,
            'IoU': iou,
            'precision': precision,
            'recall': recall,
        }, best_model_path)
      print(f"Сохранена лучшая модель | Dice: {dice:.4f} | Path: {best_model_path}")
      patience_counter = 0
    else:
      patience_counter += 1
      if patience_counter >= 4:
          print("Early stopping triggered")
          break

Для проверки адекватной работы первой модели был загружен еще один датасет с изображениями ран стоп и их масками ([Wound Image Dataset](https://data.mendeley.com/datasets/hsj38fwnvr/3)).

In [ ]:
class DFUC(Dataset):
    def __init__(self, root, transform=None):
        self.imgs = sorted((Path(root) / 'DFUC2022_train_images').glob('*.jpg'))
        self.masks = [Path(root) / 'DFUC2022_train_masks'/f.name.replace('.jpg','.png') for f in self.imgs]
        self.transform = transform
        
    def __len__(self): return len(self.imgs)
        
    def __getitem__(self, i):
        img = torch.from_numpy(np.array(Image.open(self.imgs[i]).convert('RGB'))).permute(2,0,1).float()
        mask = torch.from_numpy(np.array(Image.open(self.masks[i]).convert('L')) > 127).long()
        if self.transform:
            aug = self.transform(image=img.numpy().transpose(1,2,0), mask=mask.numpy())
            img = torch.from_numpy(aug['image'].transpose(2,0,1)).float()
            mask = torch.from_numpy(aug['mask']).long()
            
        return {'image': img, 'mask': mask}

In [ ]:
dfuc = DFUC('../data/raw/DFUC2022_train_release')

In [ ]:
transform_wound = A.Compose([
    A.Resize(512, 512, interpolation=cv2.INTER_AREA),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
], additional_targets={'mask': 'mask'})

dfuc = DFUC('../data/raw/DFUC2022_train_release', transform=transform_wound)

loader = DataLoader(dfuc, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
random.seed(42)

model = smp.DeepLabV3(encoder_name='timm-efficientnet-b0', encoder_weights=None,
                      in_channels=3, classes=1, activation=None).to(device)
checkpoint = torch.load('../src/models/baseline_deep_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])

val_loss = 0.0
tp, fp, fn, tn = 0, 0, 0, 0

model.eval()
with torch.no_grad():
    for batch in tqdm(loader, desc="Оценка"):
        x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
        logits = model(x)

        pred = torch.sigmoid(logits) > 0.5
        pred = pred.to(device).float()
        tp += (pred * y).sum().item()
        fp += (pred * (1 - y)).sum().item()
        fn += ((1 - pred) * y).sum().item()
        tn += ((1 - pred) * (1 - y)).sum().item()

dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
iou  = tp / (tp + fp + fn + 1e-6)
precision = tp / (tp + fp + 1e-6)
recall = tp / (tp + fn + 1e-6)

#### Краткая информация по датасету Wound Image Dataset

| Параметр | Значение |
|----------|----------|
| **Организация-источник** | Bangladesh University of Business and Technology |
| **Контекст сбора** | Датасет включает 8 клинически значимых типов ран нижних конечностей |
| **Лицензия** | CC BY 4.0 (коммерческое использование разрешено)|
| **Ссылка на публикацию** | https://data.mendeley.com/datasets/hsj38fwnvr/2 |
| **Общий объём** | 8 129 изображений                                   |
| **Разделение** | • Здоровые стопы - 2757 фото<br>• Стопы с ранами: 2686 пар фото + маска |
| **Формат изображений** | JPG, у фото разрешение 331–331 px, у масок 224-224 px                                               |
| **Формат масок** | PNG, одноканальные (grayscale), значения: `0` = фон, `255` = рана        |
| **Классы** | Бинарная сегментация: `0` — здоровая кожа/фон, `1` — область диабетической язвы |

**Создадим класс для нового датасета**

In [64]:
class WoundDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.img_dir = self.root / 'wound_main'
        self.mask_dir = self.root / 'wound_mask'
        
        self.imgs = sorted(list(self.img_dir.glob('wound_main-*.jpg')))
        self.valid_imgs = []
        self.masks = []
        
        for img_path in self.imgs:
            number = img_path.stem.replace('wound_main-', '')
            mask_path_jpg = self.mask_dir / f'wound_mask-{number}.jpg' # ищем маску по номеру фото
            if mask_path_jpg.exists():
                self.valid_imgs.append(img_path)
                self.masks.append(mask_path_jpg)
            else:
                print(f"Предупреждение: маска не найдена для {img_path.name}")
                
        self.transform = transform

    def __len__(self):
        return len(self.valid_imgs)
        
    def __getitem__(self, idx):
        img_path = self.valid_imgs[idx]
        mask_path = self.masks[idx]
        
        img_np = np.array(Image.open(img_path).convert('RGB'))
        mask_np = np.array(Image.open(mask_path).convert('L'))
        mask_np = cv2.resize(mask_np, (img_np.shape[1], img_np.shape[0]), interpolation=cv2.INTER_NEAREST) # меняем размеры маски под размеры фото, используем интерполяцию
        mask_np = (mask_np > 127).astype(np.uint8)
        
        if self.transform:
            augmented = self.transform(image=img_np, mask=mask_np)
            img_np = augmented['image']
            mask_np = augmented['mask']
        
        # Картинка: (H, W, C) -> (C, H, W), тип float32
        img_tensor = torch.from_numpy(img_np.transpose(2, 0, 1)).float()
        # Маска: (H, W), тип int64 (long)
        mask_tensor = torch.from_numpy(mask_np).long()
        
        return {'image': img_tensor, 'mask': mask_tensor}

In [ ]:
random.seed(42)

transform_wound = A.Compose([
    A.Resize(512, 512, interpolation=cv2.INTER_AREA),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
], additional_targets={'mask': 'mask'})

wound_dataset = WoundDataset('../data/raw/Wound Image Dataset', transform=transform_wound)

loader = DataLoader(wound_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

**Проверим лучшую модель на других данных.**

In [ ]:
model = smp.Unet(encoder_name='timm-efficientnet-b0',
                 encoder_weights=None,
                 in_channels=3, classes=1, activation=None
).to(device)
model.load_state_dict(base_model_1['model_state_dict'])

val_loss = 0.0
tp, fp, fn, tn = 0, 0, 0, 0

model.eval()
with torch.no_grad():
    for batch in tqdm(loader, desc="Оценка"):
        x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
        logits = model(x)

        pred = torch.sigmoid(logits) > 0.3
        pred = pred.to(device).float()
        tp += (pred * y).sum().item()
        fp += (pred * (1 - y)).sum().item()
        fn += ((1 - pred) * y).sum().item()
        tn += ((1 - pred) * (1 - y)).sum().item()

dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
iou  = tp / (tp + fp + fn + 1e-6)
precision = tp / (tp + fp + 1e-6)
recall = tp / (tp + fn + 1e-6)

In [ ]:
print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} |")

По полученным данным можно сделать вывод, что модель переобучилась на данных датасета `FuSeg` и обладает плохой обобщающей способностью,
показывая низкий Dice на датасете `Wound Image`. Понижение порога уверенности до 0.3 (`threshold = 0.3`) при предсказании и добавление 
механизма TTA на инференсе незначительно улучшают метрики.

**Механизм TTA для итоговых предсказаний на инференсе лучшей модели**

Здесь использовались оригинальный снимок, а также повороты стопы на 90, 180, 270 градусов, а также flip.

In [ ]:
def tta_predict(model, img_tensor, device, threshold=0.5):
    """TTA с 5 аугментациями + усреднение вероятностей"""
    preds = []
    transforms = [
        lambda x: x,
        lambda x: torch.flip(x, [-1]),
        lambda x: torch.rot90(x, k=1, dims=[-2,-1]),
        lambda x: torch.rot90(x, k=2, dims=[-2,-1]),
        lambda x: torch.rot90(x, k=3, dims=[-2,-1])
    ]
    inverses = [
        lambda x: x,
        lambda x: torch.flip(x, [-1]),
        lambda x: torch.rot90(x, k=-1, dims=[-2,-1]),
        lambda x: torch.rot90(x, k=-2, dims=[-2,-1]),
        lambda x: torch.rot90(x, k=-3, dims=[-2,-1])
    ]
    with torch.no_grad():
        for t, inv in zip(transforms, inverses):
            aug = t(img_tensor.to(device))
            prob = torch.sigmoid(model(aug)).cpu()
            preds.append(inv(prob))
    prob_map = torch.stack(preds).mean(dim=0)
    mask = (prob_map > threshold).float()
    return prob_map, mask

In [ ]:
model = smp.Unet(encoder_name='timm-efficientnet-b0',
                 encoder_weights=None,
                 in_channels=3, classes=1, activation=None
).to(device)
model.load_state_dict(base_model_1['model_state_dict'])

val_loss = 0.0
tp, fp, fn, tn = 0, 0, 0, 0

model.eval()
with torch.no_grad():
    for batch in tqdm(loader, desc="Оценка"):
        x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
        logits = model(x)

        _, pred = tta_predict(model, x, device, threshold=0.3)
        pred = pred.to(device).float()
        tp += (pred * y).sum().item()
        fp += (pred * (1 - y)).sum().item()
        fn += ((1 - pred) * y).sum().item()
        tn += ((1 - pred) * (1 - y)).sum().item()

dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
iou  = tp / (tp + fp + fn + 1e-6)
precision = tp / (tp + fp + 1e-6)
recall = tp / (tp + fn + 1e-6)

In [ ]:
print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} |")

**Таким образом, необходимо модифицировать модель для получения рабочих метрик.**

#### Пайплайн улучшения модели включает

1. Добавление более агрессивных аугментаций для учета разного характера съемки.
2. Pre-train модели на датасете `Wound Image` для улучшения обобщающей способности модели на ранах.
3. Fine-tune модели с последовательной разморозкой декодера и энкодера.
4. Итоговый инференс с использованием TTA.

Первым делом разобьем `Wound Image Dataset` на трейн и тест, используя стратифицированный сплит по размеру ран.
Для этого проанализируем датасет по размеру ран, построим график и определим количество больших и малых ран.

In [ ]:
def analyze_wound_sizes(dataset):
    """
    Анализирует размеры ран. Возвращает DataFrame с подробной информацией.
    """
    data = []  # список словарей для DataFrame

    for i in range(len(dataset)):
        sample = dataset[i]
        mask = sample['mask'].numpy()  # (H, W)
        img_path = dataset.valid_imgs[i].name  # имя файла изображения

        # Находим координаты пикселей раны (значение 1)
        coords = np.where(mask > 0)
        area_pixels = len(coords[0])  # количество пикселей раны
        image_area = mask.shape[0] * mask.shape[1]  # общая площадь изображения
        wound_ratio = area_pixels / image_area  # доля раны от всего изображения

        data.append({
            'image_name': img_path,
            'area_pixels': area_pixels,
            'wound_ratio': wound_ratio,
            })

    return pd.DataFrame(data)

In [ ]:
wound_dataset = WoundDataset('../data/raw/Wound Image Dataset')
stats = analyze_wound_sizes(wound_dataset)
stats

In [ ]:
# Гистограмма размера ран
plt.hist(stats["wound_ratio"] * 100, bins=30)
plt.xlabel("Площадь раны, %")
plt.title("Распределение размеров ран")
plt.show()

# Статистики
print(f"Медиана площади: {stats['wound_ratio'].median()*100:.2f}%")
print(f"Диапазон: {stats['wound_ratio'].min()*100:.2f}% – {stats['wound_ratio'].max()*100:.2f}%")
print(f"Квантиль 0.25: {stats['wound_ratio'].quantile(0.25):.2f}")
print(f"Квантиль 0.75: {stats['wound_ratio'].quantile(0.75):.2f}")

# Фильтрация: найти очень маленькие раны, очень большие раны
small_wounds = stats[stats["wound_ratio"] < 0.03]
print(f"Маленьких ран (<3 %): {len(small_wounds)} из {len(stats)}")
big_wounds = stats[stats["wound_ratio"] > 0.15]
print(f"Больших ран(>15 %): {len(big_wounds)} из {len(stats)}")

In [ ]:
# классифицируем раны по размеру
def size_bin(r): 
    if r < 0.03: return 0  # маленькие (<3%)
    elif r <= 0.15: return 1  # средние (3–15%)
    else: return 2  # большие (>15%)
stats["bin"] = stats["wound_ratio"].apply(size_bin)
stats

Теперь используем данные статистики для равномерного распределения размеров ран между `train`, `validation` и `test`.

In [38]:
from sklearn.model_selection import train_test_split


wound_image_dir = Path("../data/processed/wound_image_splits") # папка, куда будем сохранять данные после разбиения
wound_image_dir.mkdir(parents=True, exist_ok=True)

trainval_df, test_df = train_test_split(
    stats, 
    test_size=0.10, 
    stratify=stats['bin'],
    random_state=42
)

val_relative_size = 0.15 / (1 - 0.10)

train_df, val_df = train_test_split(
    trainval_df, 
    test_size=val_relative_size, 
    stratify=trainval_df['bin'],
    random_state=42
)

print(f"   Train: {len(train_df)} изображений ({len(train_df)/len(stats)*100:.1f}%)")
print(f"   Val:   {len(val_df)} изображений ({len(val_df)/len(stats)*100:.1f}%)")
print(f"   Test:  {len(test_df)} изображений ({len(test_df)/len(stats)*100:.1f}%)")

# сохраняем csv 
train_df[['image_name', 'area_pixels', 'wound_ratio']].to_csv(wound_image_dir / "train.csv", index=False)
val_df[['image_name', 'area_pixels', 'wound_ratio']].to_csv(wound_image_dir / "val.csv", index=False)
test_df[['image_name', 'area_pixels', 'wound_ratio']].to_csv(wound_image_dir / "test.csv", index=False)

print(f"\n✅ CSV-файлы сохранены в {wound_image_dir}")

   Train: 2014 изображений (75.0%)
   Val:   403 изображений (15.0%)
   Test:  269 изображений (10.0%)

✅ CSV-файлы сохранены в ..\data\processed\wound_image_splits


In [39]:
train_df

,image_name,area_pixels,wound_ratio,bin
256,wound_main-0257.jpg,19156,0.174843,2
1058,wound_main-1059.jpg,14016,0.127929,1
744,wound_main-0745.jpg,18015,0.164429,2
349,wound_main-0350.jpg,3741,0.034145,1
1230,wound_main-1231.jpg,2963,0.027044,0
...,...,...,...,...
2144,wound_main-2145.jpg,5612,0.051223,1
1239,wound_main-1240.jpg,25764,0.235157,2
832,wound_main-0833.jpg,348,0.003176,0
2662,wound_main-2663.jpg,13583,0.123977,1


Модифицируем класс `WoundDataset` для чтения `csv` файлов.

In [40]:
class WoundDatasetCSV(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        """
        Args:
            csv_path: путь к CSV-файлу (train.csv, val.csv или test.csv)
            root_dir: корневая папка датасета (содержит images/ и masks/)
            transform: albumentations трансформы
        """
        self.df = pd.read_csv(csv_path)
        self.root_dir = Path(root_dir)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        
        # Формируем пути
        img_path = self.root_dir / 'wound_main' / img_name
        number = img_path.stem.replace('wound_main-', '')
        mask_path = self.root_dir / 'wound_mask' / f'wound_mask-{number}.jpg'
        
        # Загрузка       
        img_np = np.array(Image.open(img_path).convert('RGB'))
        mask_np = np.array(Image.open(mask_path).convert('L'))
        mask_np = cv2.resize(mask_np, (img_np.shape[1], img_np.shape[0]), interpolation=cv2.INTER_NEAREST) # меняем размеры маски под размеры фото, используем интерполяцию
        mask_np = (mask_np > 127).astype(np.uint8)
        
        if self.transform:
            augmented = self.transform(image=img_np, mask=mask_np)
            img_np = augmented['image']
            mask_np = augmented['mask']
        
        # Картинка: (H, W, C) -> (C, H, W), тип float32
        img_tensor = torch.from_numpy(img_np.transpose(2, 0, 1)).float()
        # Маска: (H, W), тип int64 (long)
        mask_tensor = torch.from_numpy(mask_np).long()
        
        return {'image': img_tensor, 'mask': mask_tensor}

#### 1. Добавление более агрессивных аугментаций для учета разного характера съемки

In [41]:
train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.85, 1.15), rotate=(-20, 20), shear=(-5, 5), p=0.5),
    
    # Имитация domain shift (СВЕТ и ЦВЕТ)
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.7), # Компенсация плохого освещения
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
    
    # Имитация артефактов камер (Mendeley - это JPG с телефонов)
    A.GaussianBlur(blur_limit=(3, 7), p=0.4),
    A.GaussNoise(p=0.3),
    A.ImageCompression(quality_range=(75, 100), p=0.3),
    
    A.Resize(512, 512, interpolation=cv2.INTER_AREA),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
], additional_targets={'mask': 'mask'})

val_tf = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
], additional_targets={'mask':'mask'})

test_tf = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
], additional_targets={'mask':'mask'})

In [42]:
random.seed(42)

train_dataset = WoundDatasetCSV('../data/processed/wound_image_splits/train.csv',
                                '../data/raw/Wound Image Dataset/', 
                                transform=train_tf
                               )

val_dataset = WoundDatasetCSV('../data/processed/wound_image_splits/val.csv',
                                '../data/raw/Wound Image Dataset/', 
                                transform=val_tf
                               )

test_dataset = WoundDatasetCSV('../data/processed/wound_image_splits/test.csv',
                                '../data/raw/Wound Image Dataset/', 
                                transform=test_tf
                               )

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

print(f"✅ Train: {len(train_dataset)} изображений")
print(f"✅ Val:   {len(val_dataset)} изображений")
print(f"✅ Test:  {len(test_dataset)} изображений")

✅ Train: 2014 изображений
✅ Val:   403 изображений
✅ Test:  269 изображений


#### 2. Pre-train модели на датасете Wound Image для улучшения обобщающей способности модели на ранах

Обучаем лучшую модель сначала на данном датасете на 15 эпохах, этого достаточно для получения общих признаков.

In [51]:
model = smp.Unet(encoder_name='timm-efficientnet-b0', encoder_weights='imagenet', 
                 in_channels=3, classes=1, activation=None,
                 decoder_dropout=0.2).to(device)
criterion = DiceFocalLoss(dice_weight=1.0, focal_weight=0.5, 
                          focal_gamma=2.0, focal_alpha=0.25)
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

In [52]:
random.seed(42)

max_epochs = 10
best_dice = 0.0
best_val_loss = float('inf')
best_model_path = WEIGHTS_DIR / "baseline_WID_best.pth"

for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss = 0
    with tqdm(train_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Train]", leave=False) as pbar:
        for batch in pbar:
            x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Val]  ", leave=False) as pbar:
            for batch in pbar:
                x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()

                pred = torch.sigmoid(logits) > 0.5
                pred = pred.to(device).float()
                tp += (pred * y).sum().item()
                fp += (pred * (1 - y)).sum().item()
                fn += ((1 - pred) * y).sum().item()
                tn += ((1 - pred) * (1 - y)).sum().item()

    val_loss /= len(val_loader)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou  = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)

    scheduler.step(dice)


    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")
    if dice > best_dice:
      best_dice = dice
      best_val_loss = val_loss
      torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': dice,
            'val_loss': val_loss,
            'IoU': iou,
            'precision': precision,
            'recall': recall,
        }, best_model_path)
      print(f"Сохранена лучшая модель | Dice: {dice:.4f} | Path: {best_model_path}")

Epoch 01 | Train Loss: 0.8233 | Val Loss: 0.8053 | Dice: 0.5849 | IoU: 0.4134 | Precision: 0.7370 | Recall: 0.4849 | LR: 2.0e-04
Сохранена лучшая модель | Dice: 0.5849 | Path: ..\src\models\baseline_WID_best.pth


Epoch 02 | Train Loss: 0.8127 | Val Loss: 0.8056 | Dice: 0.5406 | IoU: 0.3704 | Precision: 0.8174 | Recall: 0.4038 | LR: 2.0e-04


Epoch 03 | Train Loss: 0.8101 | Val Loss: 0.8030 | Dice: 0.5899 | IoU: 0.4183 | Precision: 0.7277 | Recall: 0.4960 | LR: 2.0e-04
Сохранена лучшая модель | Dice: 0.5899 | Path: ..\src\models\baseline_WID_best.pth


Epoch 04 | Train Loss: 0.8078 | Val Loss: 0.8028 | Dice: 0.5844 | IoU: 0.4128 | Precision: 0.7836 | Recall: 0.4659 | LR: 2.0e-04


Epoch 05 | Train Loss: 0.8081 | Val Loss: 0.8011 | Dice: 0.5998 | IoU: 0.4283 | Precision: 0.7469 | Recall: 0.5011 | LR: 2.0e-04
Сохранена лучшая модель | Dice: 0.5998 | Path: ..\src\models\baseline_WID_best.pth


Epoch 06 | Train Loss: 0.8071 | Val Loss: 0.8019 | Dice: 0.6060 | IoU: 0.4348 | Precision: 0.7615 | Recall: 0.5033 | LR: 2.0e-04
Сохранена лучшая модель | Dice: 0.6060 | Path: ..\src\models\baseline_WID_best.pth


Epoch 07 | Train Loss: 0.8064 | Val Loss: 0.8012 | Dice: 0.6159 | IoU: 0.4450 | Precision: 0.7608 | Recall: 0.5173 | LR: 2.0e-04
Сохранена лучшая модель | Dice: 0.6159 | Path: ..\src\models\baseline_WID_best.pth


Epoch 08 | Train Loss: 0.8052 | Val Loss: 0.8002 | Dice: 0.6357 | IoU: 0.4659 | Precision: 0.7400 | Recall: 0.5571 | LR: 2.0e-04
Сохранена лучшая модель | Dice: 0.6357 | Path: ..\src\models\baseline_WID_best.pth


Epoch 09 | Train Loss: 0.8053 | Val Loss: 0.8027 | Dice: 0.5910 | IoU: 0.4194 | Precision: 0.7926 | Recall: 0.4711 | LR: 2.0e-04


Epoch 10 | Train Loss: 0.8035 | Val Loss: 0.7971 | Dice: 0.6732 | IoU: 0.5074 | Precision: 0.6951 | Recall: 0.6527 | LR: 2.0e-04
Сохранена лучшая модель | Dice: 0.6732 | Path: ..\src\models\baseline_WID_best.pth


#### 3. Fine-tune модели с последовательной разморозкой декодера и энкодера

In [70]:
random.seed(42)

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.85, 1.15), rotate=20, translate_percent=0.05, p=0.5), # афинные преобразования (масштабирование 85-115 %, поворот на 15 градусов, сдвигает изображение на 5 % от его размера)
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),                  # улучшает локальный контраст изображения методом CLAHE (Contrast Limited Adaptive Histogram Equalization)
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.4), # случайным образом изменяет цветовые характеристики изображения (яркость, контрастность, насыщенность, оттенок)
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
], additional_targets={'mask': 'mask'})

val_tf = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
], additional_targets={'mask':'mask'})

train_ds = FUSegBaseline(DATASET_ROOT, 'train', train_tf)
val_ds   = FUSegBaseline(DATASET_ROOT, 'validation', val_tf)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
# Инициализируем ту же архитектуру
model = smp.Unet(
    encoder_name='timm-efficientnet-b0',
    encoder_weights=None,
    in_channels=3, classes=1, activation=None,
    decoder_dropout=0.2
).to(device)

checkpoint = torch.load('../src/models/baseline_WID_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

for param in model.encoder.parameters():
    param.requires_grad = False

criterion = DiceFocalLoss(dice_weight=1.0, focal_weight=0.5, 
                          focal_gamma=2.0, focal_alpha=0.75)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=2e-4, weight_decay=5e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

In [74]:
random.seed(42)

max_epochs = 15
best_dice_s1 = 0.0
patience_counter = 0

for epoch in range(1, max_epochs + 1): # обучаем декодер на 15 эпохах
    model.train()
    train_loss = 0.0
    with tqdm(train_loader, desc=f"Epoch {epoch:02d}/ {max_epochs} [Train]", leave=False) as pbar:
        for batch in pbar:
            x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= len(train_loader)
    
    model.eval()
    val_loss = 0.0
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch:02d}/ {max_epochs} [Val]  ", leave=False) as pbar:
            for batch in pbar:
                x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()

                pred = torch.sigmoid(logits) > 0.5
                pred = pred.to(device).float()
                tp += (pred * y).sum().item()
                fp += (pred * (1 - y)).sum().item()
                fn += ((1 - pred) * y).sum().item()
                tn += ((1 - pred) * (1 - y)).sum().item()

    val_loss /= len(val_loader)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou  = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)

    scheduler.step(dice)
    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")
    
    if dice > best_dice_s1:
        best_dice_s1 = dice
        torch.save(model.state_dict(), "../src/models/decoder_step.pth")
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print("Early stopping triggered")
            break

Epoch 01 | Train Loss: 0.9727 | Val Loss: 0.9688 | Dice: 0.6089 | IoU: 0.4377 | Precision: 0.4460 | Recall: 0.9590 | LR: 2.0e-04


Epoch 02 | Train Loss: 0.9732 | Val Loss: 0.9690 | Dice: 0.5825 | IoU: 0.4109 | Precision: 0.4163 | Recall: 0.9691 | LR: 2.0e-04


Epoch 03 | Train Loss: 0.9730 | Val Loss: 0.9689 | Dice: 0.7500 | IoU: 0.6001 | Precision: 0.6595 | Recall: 0.8694 | LR: 2.0e-04


Epoch 04 | Train Loss: 0.9731 | Val Loss: 0.9687 | Dice: 0.7527 | IoU: 0.6035 | Precision: 0.6393 | Recall: 0.9151 | LR: 2.0e-04


Epoch 05 | Train Loss: 0.9728 | Val Loss: 0.9684 | Dice: 0.7154 | IoU: 0.5569 | Precision: 0.5789 | Recall: 0.9361 | LR: 2.0e-04


Epoch 06 | Train Loss: 0.9726 | Val Loss: 0.9685 | Dice: 0.6491 | IoU: 0.4805 | Precision: 0.4907 | Recall: 0.9585 | LR: 2.0e-04


Epoch 07 | Train Loss: 0.9721 | Val Loss: 0.9696 | Dice: 0.5109 | IoU: 0.3431 | Precision: 0.3455 | Recall: 0.9806 | LR: 2.0e-04


Epoch 08 | Train Loss: 0.9727 | Val Loss: 0.9682 | Dice: 0.6925 | IoU: 0.5296 | Precision: 0.5416 | Recall: 0.9599 | LR: 1.0e-04
Early stopping triggered


In [75]:
random.seed(42)
max_epochs = 15

print("Шаг 2: Тонкая настройка всей модели (Дифференциальный LR)")
model.load_state_dict(torch.load("../src/models/decoder_step.pth"))
best_model_path = WEIGHTS_DIR / "baseline_fusseg_best.pth"

for param in model.encoder.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 1e-5},
    {"params": model.decoder.parameters(), "lr": 1e-4},  # Стандартно
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_dice_final = 0.0
patience_counter = 0

for epoch in range(1, 16): # всю модель дообучаем на 15 эпохах
    model.train()
    train_loss = 0.0
    with tqdm(train_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Train]", leave=False) as pbar:
        for batch in pbar:
            x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= len(train_loader)
    
    model.eval()
    val_loss = 0.0
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Val]  ", leave=False) as pbar:
            for batch in pbar:
                x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()

                pred = torch.sigmoid(logits) > 0.5
                pred = pred.to(device).float()
                tp += (pred * y).sum().item()
                fp += (pred * (1 - y)).sum().item()
                fn += ((1 - pred) * y).sum().item()
                tn += ((1 - pred) * (1 - y)).sum().item()

    val_loss /= len(val_loader)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou  = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)

    scheduler.step(dice)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")
    
    if dice > best_dice_final:
      best_dice_final = dice
      best_val_loss = val_loss
      torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': dice,
            'val_loss': val_loss,
            'IoU': iou,
            'precision': precision,
            'recall': recall,
        }, best_model_path)
      print(f"Сохранена лучшая модель | Dice: {dice:.4f} | Path: {best_model_path}")
      patience_counter = 0
    else:
      patience_counter += 1
      if patience_counter >= 5:
          print("Early stopping triggered")
          break

C:\Users\Андрей\AppData\Local\Temp\ipykernel_21780\3173991301.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../src/models/decoder_ste

Шаг 2: Тонкая настройка всей модели (Дифференциальный LR)


Epoch 01 | Train Loss: 0.9721 | Val Loss: 0.9683 | Dice: 0.6867 | IoU: 0.5229 | Precision: 0.5356 | Recall: 0.9568 | LR: 1.0e-05
Сохранена лучшая модель | Dice: 0.6867 | Path: ..\src\models\baseline_fusseg_best.pth


Epoch 02 | Train Loss: 0.9732 | Val Loss: 0.9686 | Dice: 0.7743 | IoU: 0.6317 | Precision: 0.6818 | Recall: 0.8959 | LR: 1.0e-05
Сохранена лучшая модель | Dice: 0.7743 | Path: ..\src\models\baseline_fusseg_best.pth


Epoch 03 | Train Loss: 0.9723 | Val Loss: 0.9680 | Dice: 0.7245 | IoU: 0.5680 | Precision: 0.5832 | Recall: 0.9561 | LR: 1.0e-05


Epoch 04 | Train Loss: 0.9720 | Val Loss: 0.9682 | Dice: 0.6639 | IoU: 0.4969 | Precision: 0.5046 | Recall: 0.9702 | LR: 1.0e-05


Epoch 05 | Train Loss: 0.9718 | Val Loss: 0.9680 | Dice: 0.7646 | IoU: 0.6189 | Precision: 0.6432 | Recall: 0.9423 | LR: 1.0e-05


Epoch 06 | Train Loss: 0.9720 | Val Loss: 0.9680 | Dice: 0.7688 | IoU: 0.6245 | Precision: 0.6482 | Recall: 0.9446 | LR: 5.0e-06


Epoch 07 | Train Loss: 0.9713 | Val Loss: 0.9679 | Dice: 0.7340 | IoU: 0.5798 | Precision: 0.5940 | Recall: 0.9605 | LR: 5.0e-06
Early stopping triggered


In [83]:
model = smp.Unet(encoder_name='timm-efficientnet-b0',
                 encoder_weights=None,
                 in_channels=3, classes=1, activation=None
).to(device)
checkpoint = torch.load('../src/models/baseline_fusseg_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])

val_loss = 0.0
tp, fp, fn, tn = 0, 0, 0, 0

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Оценка"):
        x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
        logits = model(x)

        pred = torch.sigmoid(logits) > 0.25
        pred = pred.to(device).float()
        tp += (pred * y).sum().item()
        fp += (pred * (1 - y)).sum().item()
        fn += ((1 - pred) * y).sum().item()
        tn += ((1 - pred) * (1 - y)).sum().item()

dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
iou  = tp / (tp + fp + fn + 1e-6)
precision = tp / (tp + fp + 1e-6)
recall = tp / (tp + fn + 1e-6)

C:\Users\Андрей\AppData\Local\Temp\ipykernel_21780\4154679287.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('../src/models/baseline_fusseg_best

In [84]:
print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} |")

Dice: 0.4635 | IoU: 0.3017 | Precision: 0.6585 | Recall: 0.3576 |


In [85]:
model = smp.Unet(encoder_name='timm-efficientnet-b0',
                 encoder_weights=None,
                 in_channels=3, classes=1, activation=None
).to(device)
checkpoint = torch.load('../src/models/baseline_fusseg_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])

val_loss = 0.0
tp, fp, fn, tn = 0, 0, 0, 0

model.eval()
with torch.no_grad():
    for batch in tqdm(loader, desc="Оценка"):
        x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
        logits = model(x)

        _, pred = tta_predict(model, x, device, threshold=0.25)
        pred = pred.to(device).float()
        tp += (pred * y).sum().item()
        fp += (pred * (1 - y)).sum().item()
        fn += ((1 - pred) * y).sum().item()
        tn += ((1 - pred) * (1 - y)).sum().item()

dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
iou  = tp / (tp + fp + fn + 1e-6)
precision = tp / (tp + fp + 1e-6)
recall = tp / (tp + fn + 1e-6)

C:\Users\Андрей\AppData\Local\Temp\ipykernel_21780\684332198.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('../src/models/baseline_fusseg_best.

In [86]:
print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} |")

Dice: 0.4796 | IoU: 0.3154 | Precision: 0.6036 | Recall: 0.3979 |
